In [ ]:
#!pip install transformers datasets

# Data Loading

In [1]:
with open('data/small_vocab_en', 'r') as f:
    eng_sentences = f.read().split('\n')
    
with open('data/small_vocab_fr', 'r') as f:
    fre_sentences = f.read().split('\n')

print('Dataset Loaded')


Dataset Loaded


In [2]:
import collections
import numpy as np

print(eng_sentences[0:5],"...eng sentences...")
print(fre_sentences[0:5],"...french sentences..")
print(len(eng_sentences),"...length of english sentences...")
print(len(fre_sentences),"...length of french sentences....")

['new jersey is sometimes quiet during autumn , and it is snowy in april .', 'the united states is usually chilly during july , and it is usually freezing in november .', 'california is usually quiet during march , and it is usually hot in june .', 'the united states is sometimes mild during june , and it is cold in september .', 'your least liked fruit is the grape , but my least liked is the apple .'] ...eng sentences...
["new jersey est parfois calme pendant l' automne , et il est neigeux en avril .", 'les Ã©tats-unis est gÃ©nÃ©ralement froid en juillet , et il gÃ¨le habituellement en novembre .', 'california est gÃ©nÃ©ralement calme en mars , et il est gÃ©nÃ©ralement chaud en juin .', 'les Ã©tats-unis est parfois lÃ©gÃ¨re en juin , et il fait froid en septembre .', 'votre moins aimÃ© fruit est le raisin , mais mon moins aimÃ© est la pomme .'] ...french sentences..
137861 ...length of english sentences...
137861 ...length of french sentences....


# Data Preprocessing

In [3]:
import numpy as np
import io
import unicodedata
import re
from tqdm import tqdm

def unicode_to_ascii(s) :
  """
  Unicode to ascii conversion
  """
  return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def cleanhtml(raw_html) :
  """
  Function to clean html tags and numbers
  """
  cleanr = re.compile('<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});')
  cleantext = re.sub(cleanr, '', raw_html)
  return cleantext

def cleanString(incomingString):
    """
      Function to clean unwanted symbol from text
    """
    newstring = incomingString
    newstring = newstring.replace("!","")
    newstring = newstring.replace("@","")
    newstring = newstring.replace("#","")
    newstring = newstring.replace("$","")
    newstring = newstring.replace("%","")
    newstring = newstring.replace("^","")
    newstring = newstring.replace("&","and")
    newstring = newstring.replace("*","")
    newstring = newstring.replace("(","")
    newstring = newstring.replace(")","")
    newstring = newstring.replace("+","")
    newstring = newstring.replace("=","")
    newstring = newstring.replace("?","")
    newstring = newstring.replace("\'","")
    newstring = newstring.replace("\"","")
    newstring = newstring.replace("{","")
    newstring = newstring.replace("}","")
    newstring = newstring.replace("[","")
    newstring = newstring.replace("]","")
    newstring = newstring.replace("<","")
    newstring = newstring.replace(">","")
    newstring = newstring.replace("~","")
    newstring = newstring.replace("`","")
    newstring = newstring.replace(":","")
    newstring = newstring.replace(";","")
    newstring = newstring.replace("|","")
    newstring = newstring.replace("\\","")
    newstring = newstring.replace("/","")     
    return ' '.join(newstring.split())

def preprocess_string(data) :
  """
  This function calls other
  preprocessing function for
  cleaning data
  """
  data = unicode_to_ascii(data)
  #Remove html
  data = cleanhtml(data)
  #Remove unwanted symbols
  data = cleanString(data)
  return data


def start_preprocessing(data,lang):
    print("..started preprocessing..."+lang)
    preproc_data_list = []
    for val in tqdm(data):
        preproc_data=preprocess_string(val)
        preproc_data_list.append(preproc_data)
    return preproc_data_list    

In [4]:
eng_preproc_sentence = start_preprocessing(eng_sentences,"eng")
fre_preproc_senetence = start_preprocessing(fre_sentences,"french")

..started preprocessing...eng


100%|███████████████████████████████████████████████████████████████████████| 137861/137861 [00:04<00:00, 30433.92it/s]


..started preprocessing...french


100%|███████████████████████████████████████████████████████████████████████| 137861/137861 [00:05<00:00, 26497.15it/s]


In [5]:
# Putting the start and end words in the french sentances

eng_preproc_sentence = [x.lower() for x in eng_preproc_sentence]
fre_preproc_senetence = [x.lower() for x in fre_preproc_senetence]
#fre_preproc_senetence = ["start " + x + " end" for x in fre_preproc_senetence ]

In [6]:
eng_preproc_sentence = eng_preproc_sentence[0:500] 
fre_preproc_senetence = fre_preproc_senetence[0:500] 

In [7]:
from sklearn.model_selection import train_test_split
X=eng_preproc_sentence
Y=fre_preproc_senetence
X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size = 0.1)
len(X_train),len(y_train), len(X_test), len(y_test)

(450, 450, 50, 50)

In [8]:
print(X_train[1], y_train[1])
print("......................................")
print(X_test[1],  y_test[1])

new jersey is usually quiet during fall , but it is usually warm in april . new jersey est ga©na©ralement calme au cours de l automne , mais il est ga©na©ralement chaud en avril .
......................................
she dislikes apples , peaches , and bananas . elle da©teste les pommes , les paªches et les bananes .


In [9]:
def Max_length(data):
  max_length_ = max([len(x.split(' ')) for x in data])
  return max_length_

#Training data
max_length_english = Max_length(X_train)
max_length_french = Max_length(y_train)

#Test data
max_length_english_test = Max_length(X_test)
max_length_french_test = Max_length(y_test)

print(max_length_english_test,"...max english length test..")
print(max_length_french_test,"....max french  length test..")

print(max_length_english,"...max english length train..")
print(max_length_french,"....max french  length train..")

17 ...max english length test..
18 ....max french  length test..
17 ...max english length train..
21 ....max french  length train..


In [10]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from transformers import BartForConditionalGeneration, BartTokenizer
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForSeq2Seq
import numpy as np
from sklearn.model_selection import train_test_split
import time
import datetime

C:\Users\utsav\anaconda3\envs\mlep-w1-lab\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'

In [12]:
model_name = 't5-small'
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [13]:
from datasets import Dataset
def preprocess_data(source_texts, target_texts):
    """Tokenize and prepare the data in Hugging Face Dataset format."""
    inputs = [f"translate English to French: {text}" for text in source_texts]  # Add task-specific prefix
    targets = target_texts

    # Tokenize inputs and targets
    model_inputs = tokenizer(inputs, max_length=17, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=21, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs



In [14]:
train_data = preprocess_data(X_train, y_train)
test_data = preprocess_data(X_test, y_test)

# Create Hugging Face Dataset objects
train_dataset = Dataset.from_dict(train_data)
test_dataset = Dataset.from_dict(test_data)

C:\Users\utsav\anaconda3\envs\mlep-w1-lab\lib\site-packages\transformers\tokenization_utils_base.py:3946: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [16]:
train_dataset[0]

{'input_ids': [13959,
  1566,
  12,
  2379,
  10,
  20576,
  19,
  1664,
  3,
  28013,
  383,
  10556,
  3,
  6,
  11,
  34,
  1],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [50,
  3,
  5675,
  15,
  259,
  11084,
  9030,
  7,
  835,
  17,
  3,
  35,
  8113,
  3,
  6,
  3,
  15,
  17,
  3,
  173,
  1]}

In [19]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=10,
)

# Step 5: Define the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)

# Step 6: Train the model
trainer.train()

# Step 7: Save the model and tokenizer
model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")

C:\Users\utsav\anaconda3\envs\mlep-w1-lab\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,1.665500,1.681001
2,1.192000,1.231183
3,1.314900,1.099919


('./final_model\\tokenizer_config.json',
 './final_model\\special_tokens_map.json',
 './final_model\\spiece.model',
 './final_model\\added_tokens.json')

In [20]:
# Evaluate the model
results = trainer.evaluate()
#logger.info(results)
print(results)

{'eval_loss': 1.0999187231063843, 'eval_runtime': 2.044, 'eval_samples_per_second': 24.462, 'eval_steps_per_second': 12.231, 'epoch': 3.0}


In [22]:
test_results = trainer.evaluate()
print(f"Test results: {test_results}")

Test results: {'eval_loss': 1.0999187231063843, 'eval_runtime': 2.1369, 'eval_samples_per_second': 23.399, 'eval_steps_per_second': 11.699, 'epoch': 3.0}


In [21]:
#trainer.save_model("sample-op-peft")

# Inferrence without fine-tuning

In [25]:
new_sentences = "Translate to french: I love you ." 

# Tokenize the new sentences
inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True, max_length=128)
print(inputs)
# Generate predictions
outputs = model.generate(inputs['input_ids'].to('cpu'), max_length=128)

# Decode the predictions
predictions = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
print(predictions)

{'input_ids': tensor([[30355,    15,    12, 20609,    10,    27,   333,    25,     3,     5,
             1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
['Translate to french:']


# Inference with API result

In [171]:
# Inference Using Lora
#model_name = 't5-small'
#tokenizer = T5Tokenizer.from_pretrained(model_name)
#model = T5ForConditionalGeneration.from_pretrained(model_name)
from peft import PeftModel
from peft import PeftModel, PeftConfig
#peft_model_base = T5ForConditionalGeneration.from_pretrained('t5-small')
tokenizer = T5Tokenizer.from_pretrained('t5-small')

#peft_model = PeftModel.from_pretrained(peft_model_base, 
                                      #"sample-op-peft",
                                      #torch_dtype=torch.bfloat16,
                                      #is_trainable=False)
#model = peft_model
#new_sentences = "Translate english to french: I love you"
#model.eval()

#model = T5ForConditionalGeneration.from_pretrained("sample-op-peft")
config = PeftConfig.from_pretrained("sample-op-peft")
model = T5ForConditionalGeneration.from_pretrained(config.base_model_name_or_path)
tokenizer = T5Tokenizer.from_pretrained(config.base_model_name_or_path)
# model = get_peft_model(model, peft_config)  # Apply LoRA configuration again
# model.eval()
model = PeftModel.from_pretrained(model, "sample-op-peft")
model.eval()

new_sentences = "Translate to French: I love you ." 

# Tokenize the new sentences
inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True, max_length=128)
print(inputs)
# Generate predictions
outputs = model.generate(inputs['input_ids'].to('cpu'), max_length=128)

# Decode the predictions
predictions = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
print(predictions)

# Print the predictions
# for sentence, translation in zip(new_sentences, predictions):
#     print(f"Input: {sentence}")
#     print(f"Translation: {translation}\n")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


{'input_ids': tensor([[30355,    15,    12,  2379,    10,    27,   333,    25,     3,     5,
             1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
['Je vous aime.']


# Inference with full finetuning

In [31]:
# Inference of huggin face
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load the saved model and tokenizer
model = T5ForConditionalGeneration.from_pretrained("final_model")
tokenizer = T5Tokenizer.from_pretrained('t5-small')
# text= [
#     "translate English to French : What is your name?",
#     "translate English to French: I love you",
#     "translate English to French: Where is the bathroom?",
# ]
for i in range(3):
    v = X_test[i]
    v = "translate English to French : "+v
    input_ids = tokenizer.encode(v, return_tensors="pt")
    outputs = model.generate(input_ids=input_ids, max_length=50, num_beams=4, early_stopping=True)

    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("English..."+v)
    print("Orginal: "+str(y_test[i]))
    print("Translated:", decoded_output)
    print("...........................................")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


English...translate English to French : the united states is sometimes beautiful during april , but it is usually busy in october .
Orginal: les a©tats-unis est parfois belle en avril , mais il est ga©na©ralement occupa© en octobre .
Translated: les atats-unis est parfois beau en avril, mais il est habituellement occupé en octobre.
...........................................
English...translate English to French : she disliked a little blue truck .
Orginal: elle naimait pas un petit camion bleu .
Translated: elle a aigu un petit camion bleu.
...........................................
English...translate English to French : they dislike grapes , mangoes , and limes .
Orginal: ils naiment pas les raisins , mangues et citrons verts .
Translated: ils naissent pas les raisins, les mangues et les chaux.
...........................................


In [29]:
y_test[0]

'les a©tats-unis est parfois belle en avril , mais il est ga©na©ralement occupa© en octobre .'